<a href="https://colab.research.google.com/github/orlaith-hughes/gaa-two-point-era/blob/main/two_point_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)

In [9]:
url = "https://raw.githubusercontent.com/orlaith-hughes/gaa-two-point-era/refs/heads/main/data/mayo_2026_matches.csv"
df = pd.read_csv(url)
df

,#
0,Date
1,Match and source report
2,Mayo two-pointers
3,Opponent two-pointers
4,1
5,Sat 11 Apr
6,London v Mayo (RTÉ)
7,7
8,3
9,2


In [10]:
df.info()
print()
print("Blank cells per column:")
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   #       44 non-null     object
dtypes: object(1)
memory usage: 484.0+ bytes

Blank cells per column:
#    0
dtype: int64


In [11]:
# Total score = goals x 3 + points (two-pointers are already inside points)
df["mayo_total"] = df["mayo_goals"] * 3 + df["mayo_points"]
df["opp_total"] = df["opp_goals"] * 3 + df["opp_points"]
df["margin"] = df["mayo_total"] - df["opp_total"]

# How many points two-pointers were worth, and their share of Mayo's total
df["mayo_2pt_value"] = df["mayo_2pt_made"] * 2
df["mayo_2pt_share_%"] = (df["mayo_2pt_value"] / df["mayo_total"] * 100).round(1)

# Accuracy from outside the arc (only where attempts were reported)
df["mayo_2pt_accuracy_%"] = (df["mayo_2pt_made"] / df["mayo_2pt_attempts"] * 100).round(1)

df[["stage", "opponent", "mayo_total", "opp_total", "margin",
    "mayo_2pt_made", "mayo_2pt_share_%", "mayo_2pt_accuracy_%"]]

KeyError: 'mayo_goals'

In [ ]:
known = df.dropna(subset=["mayo_2pt_made"])   # only matches where two-pointers are recorded

total_points = known["mayo_total"].sum()
total_2pt_value = known["mayo_2pt_value"].sum()

print(f"Matches with two-pointer data: {len(known)} of {len(df)}")
print(f"Points from two-pointers: {total_2pt_value:.0f} of {total_points:.0f} "
      f"({total_2pt_value / total_points * 100:.1f}%)")
print(f"Average two-pointers per match: {known['mayo_2pt_made'].mean():.1f}")

In [ ]:
chart = known.copy()
chart["goal_pts"] = chart["mayo_goals"] * 3
chart["one_pointers"] = chart["mayo_points"] - chart["mayo_2pt_value"]
labels = chart["opponent"]

fig, ax = plt.subplots()
ax.bar(labels, chart["one_pointers"], label="1-point scores")
ax.bar(labels, chart["mayo_2pt_value"], bottom=chart["one_pointers"], label="Two-pointers")
ax.bar(labels, chart["goal_pts"],
       bottom=chart["one_pointers"] + chart["mayo_2pt_value"], label="Goals")
ax.set_title("Where Mayo's points came from, 2026 championship")
ax.set_ylabel("Points")
ax.legend()
plt.tight_layout()
plt.savefig("points_breakdown.png", dpi=150)
plt.show()

In [ ]:
both = df.dropna(subset=["mayo_2pt_made", "opp_2pt_made"]).copy()
x = list(range(len(both)))
width = 0.4

fig, ax = plt.subplots()
ax.bar([i - width / 2 for i in x], both["mayo_2pt_made"], width, label="Mayo")
ax.bar([i + width / 2 for i in x], both["opp_2pt_made"], width, label="Opponent")
ax.set_xticks(x)
ax.set_xticklabels(both["opponent"])
ax.set_title("Two-pointers scored: Mayo v opponents")
ax.set_ylabel("Two-pointers")
ax.legend()
plt.tight_layout()
plt.savefig("two_pointers_vs_opponents.png", dpi=150)
plt.show()

In [ ]:
both["mayo_old"] = both["mayo_total"] - both["mayo_2pt_made"]
both["opp_old"] = both["opp_total"] - both["opp_2pt_made"]
both["old_margin"] = both["mayo_old"] - both["opp_old"]
both["result_changes"] = (both["margin"] > 0) != (both["old_margin"] > 0)

print(both[["opponent", "margin", "old_margin", "result_changes"]])

fig, ax = plt.subplots()
ax.bar([i - width / 2 for i in x], both["margin"], width, label="Actual margin (new rules)")
ax.bar([i + width / 2 for i in x], both["old_margin"], width, label="Margin under old rules")
ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(both["opponent"])
ax.set_title("Would Mayo have won under the old one-point rules?")
ax.set_ylabel("Winning margin (points)")
ax.legend()
plt.tight_layout()
plt.savefig("old_rules_comparison.png", dpi=150)
plt.show()

# How much did the two-point score decide Mayo's 2026 All-Ireland?
This notebook measures how many of Mayo's points came from two-pointers
in the 2026 championship, and whether Mayo would have won under the old rules.